# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library in Python.

### Dataset Source
The dataset is described using a Croissant schema, available at the link below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset title and description
print(dataset.metadata.name)
print(dataset.metadata.description)
# (For full metadata access: `dataset.metadata` is a structured Python object, not a dict.)

## 2. Data Overview
List the available record sets, their `@id` values, and associated fields for reference. In Croissant, record sets define tables or objects containing records, and fields correspond to the columns or properties in those objects.

In [ ]:
# List available record sets and their fields
print("Available Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs.id} | Name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()
# Save list of record set @id's for later use
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. All references use unique `@id` identifiers, as required by the Croissant specification.

We'll extract the main protein record set as an example, but you can extend the following code to other available record sets.

In [ ]:
# Extract all record sets into DataFrames using their @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} with {len(df)} rows and columns: {df.columns.tolist()}")

# Choose the main protein record set by inspecting the IDs. For this dataset, the main record set often includes 'protein' or similar keyword.
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'protein' in (rs.name or '').lower():
        main_record_set_id = rs.id
        break
# If not found, just use the first record set
if main_record_set_id is None and record_set_ids:
    main_record_set_id = record_set_ids[0]

print(f"\nUsing main record set: {main_record_set_id}\nColumns available:")
print(dataframes[main_record_set_id].columns.tolist())

# Display first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now, let's work with the main record set. We'll select a numeric field (e.g., abundance, molecular weight, peptide counts) using its unique `@id`.

We'll filter the data, normalize one field, and group by a categorical field for summary statistics.

In [ ]:
# Identify a numeric field (by @id or column name) for analysis.
# To discover available fields, show all columns:
df = dataframes[main_record_set_id]
print("Available columns:", df.columns.tolist())

# Suppose the dataset includes numeric fields such as 'Molecular Weight', 'Abundance', 'Normalized Abundance', or 'PeptideCount'.
# Let's use 'MolecularWeight' or similar field, if available. Adjust below according to your data.

# Try common column candidates for numeric analysis
possible_numeric = [c for c in df.columns if any(x in c.lower() for x in ['weight','mw','abundance','count','coverage','intensity'])]
if possible_numeric:
    numeric_field = possible_numeric[0]
else:
    # If none found, use first numeric column
    numeric_field = df.select_dtypes(include='number').columns[0]

print(f"Using numeric field: {numeric_field}")
threshold = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

# Group by a categorical field (e.g., if there is a 'Sample', 'Condition', or 'Category' column)
possible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['sample','condition','category','group','organism'])]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped statistics:\n", grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field, and, if a group field exists, compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If grouping field detected, plot mean normalized numeric field by group
if 'group_field' in locals():
    plt.figure(figsize=(9,4))
    sns.barplot(x=grouped_df.index, y=numeric_field + "_normalized", data=grouped_df.reset_index())
    plt.title(f"Mean Normalized {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean Normalized {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- The dataset provides comprehensive protein-level data from human mast cell experiments, supplied in FAIR-compliant Croissant format.
- You can programmatically access every record set, field, and value by unique `@id`.
- Further analysis can include assessing correlations between protein abundance and modifications, subgroup comparison, or advanced modeling.

<p style="color:gray">Notebook generated for demo purposes. For more information, see the [FAIR<sup>2</sup> Dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa).</p>